# Automated IRIS-AIA co-alignment

This notebook demonstrates the batch workflow developed for the MSc thesis. It matches IRIS slit-jaw images to the closest AIA 1600 Å frames in time, aligns every pair spatially, and writes the fitted shifts, uncertainties, rotation angles, and fit metric to a TSV file.

The observational FITS files are expected to be available locally.

## Observation list

The observation list uses one row per alignment window and exactly three columns:

| obsid | start_time | end_time |
|---|---|---|
| 20140329_140938_3860258481 | 2014-03-29T17:45:36 | 2014-03-29T17:51:41 |

Timestamps must use ISO format, `YYYY-MM-DDTHH:MM:SS`. A sample file is provided at `sample_data/observation_list.csv`.

In [ ]:
import gc
import sys
from pathlib import Path

import irisreader as ir
import numpy as np
import pandas as pd
from astropy.time import Time
from joblib import Parallel, delayed
from sunpy.map import Map
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils.py").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or examples directory.")
sys.path.insert(0, str(PROJECT_ROOT))

from utils import (
    align_aia_iris,
    find_matching_frames,
    iris_to_sunpy_map,
    write_error_row,
    write_to_file,
)

## Configuration

Set the paths for your local observation list, FITS data, and generated results. Each observation must be stored in a subdirectory named after its IRIS observation ID. AIA files are identified by the pattern `*.image.fits`.

In [ ]:
OBSERVATION_LIST = PROJECT_ROOT / "sample_data/observation_list.csv"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_FILE = PROJECT_ROOT / "outputs/iris_aia_alignment.tsv"

ALIGNMENT_METHOD = "chi2"  # Either "chi2" or "phase"
N_JOBS = -1
MAX_TIME_DIFFERENCE_SECONDS = 24

In [ ]:
REQUIRED_COLUMNS = {"obsid", "start_time", "end_time"}

observation_table = pd.read_csv(OBSERVATION_LIST, dtype={"obsid": "string"})
missing_columns = REQUIRED_COLUMNS.difference(observation_table.columns)
unexpected_columns = set(observation_table.columns).difference(REQUIRED_COLUMNS)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
if unexpected_columns:
    raise ValueError(f"Unexpected columns: {sorted(unexpected_columns)}")

observation_table = observation_table.loc[:, ["obsid", "start_time", "end_time"]].copy()
observation_table["obsid"] = observation_table["obsid"].str.strip()
for column in ("start_time", "end_time"):
    observation_table[column] = pd.to_datetime(
        observation_table[column], errors="raise", format="%Y-%m-%dT%H:%M:%S"
    )

if observation_table.isna().any().any():
    raise ValueError("The observation list contains missing values.")
if observation_table["obsid"].eq("").any():
    raise ValueError("Every row must contain an IRIS observation ID.")
if (observation_table["end_time"] < observation_table["start_time"]).any():
    raise ValueError("Every end_time must be equal to or later than start_time.")

jobs = [
    {
        "obs_id": row.obsid,
        "start": row.start_time.strftime("%Y-%m-%dT%H:%M:%S"),
        "end": row.end_time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    for row in observation_table.itertuples(index=False)
]

print(f"Loaded {len(jobs)} alignment window(s) from {OBSERVATION_LIST}.")

## Batch alignment

For each time window, the closest AIA frame is selected for every IRIS frame within the configured tolerance. Results are appended to the TSV output. Failed observations are recorded with a descriptive status row so that the remaining batch can continue.

In [ ]:
if ALIGNMENT_METHOD not in {"chi2", "phase"}:
    raise ValueError('ALIGNMENT_METHOD must be either "chi2" or "phase".')
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Observation directory not found: {DATA_DIR.resolve()}")

for job in tqdm(jobs, desc="Observations"):
    obs_id = job["obs_id"]
    obs_folder = DATA_DIR / obs_id

    if not obs_folder.is_dir():
        message = f"Observation directory not found: {obs_folder}"
        write_error_row(OUTPUT_FILE, obs_id, message)
        print(f"{obs_id}: {message}")
        continue

    aia_files = sorted(obs_folder.glob("*.image.fits"))
    if not aia_files:
        message = f"No AIA files matching *.image.fits found in {obs_folder}"
        write_error_row(OUTPUT_FILE, obs_id, message)
        print(f"{obs_id}: {message}")
        continue

    try:
        observation = ir.observation(str(obs_folder))
        iris_sji = observation.sji[0]
        aia_maps = Map(aia_files, sequence=True, allow_errors=True)

        iris_times_s = np.asarray(iris_sji.get_timestamps())
        aia_times_s = np.asarray([map_.date.unix for map_ in aia_maps])
        matching_frames = find_matching_frames(
            iris_times_s,
            aia_times_s,
            Time(job["start"]).unix,
            Time(job["end"]).unix,
            delta_t=MAX_TIME_DIFFERENCE_SECONDS,
        )

        if not matching_frames:
            message = "No matching AIA and IRIS frames found."
            write_error_row(OUTPUT_FILE, obs_id, message)
            print(f"{obs_id}: {message}")
            continue

        matched_maps = []
        for aia_frame, iris_frame in matching_frames:
            matched_maps.append(
                (aia_maps[aia_frame], iris_to_sunpy_map(iris_sji, iris_frame))
            )

        results = Parallel(n_jobs=N_JOBS, backend="threading")(
            delayed(align_aia_iris)(
                aia_map.data,
                aia_map.meta,
                iris_map.data,
                iris_map.meta,
                method=ALIGNMENT_METHOD,
            )
            for aia_map, iris_map in matched_maps
        )

        write_to_file(
            out_file=OUTPUT_FILE,
            observation_id=obs_id,
            matches=matching_frames,
            results=results,
            iris_times_s=iris_times_s,
            aia_times_s=aia_times_s,
        )

        print(f"{obs_id}: aligned {len(matching_frames)} frame pair(s).")

    except Exception as error:
        write_error_row(OUTPUT_FILE, obs_id, str(error))
        print(f"{obs_id}: alignment failed: {error}")

    finally:
        gc.collect()

print(f"Finished. Results were written to {OUTPUT_FILE}.")